In [ ]:
!pip uninstall -y transformers -q

!pip install -q \
    git+https://github.com/huggingface/transformers.git@096f25ae1f501a084d8ff2dcaf25fbc2bd60eba4 \
    accelerate \
    pandas \
    tqdm

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 110.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import os
import gc
import time

import torch
import pandas as pd

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

In [ ]:
import transformers

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Transformers: 4.52.0.dev0
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


In [ ]:
CSV_PATH = "/content/output_with_bullet_points.csv"

eval_df = pd.read_csv(CSV_PATH)

print("Rows:", len(eval_df))
print("Columns:", eval_df.columns.tolist())

eval_df.head()

Rows: 500
Columns: ['text', 'source', 'example_id', 'bullet_points']


,text,source,example_id,bullet_points
0,Powell: N. Korea Blast Not Nuclear Event The U...,ag_news,0,- Powell says the large North Korean explosion...
1,Jerusalem (CNN) -- Two attacks carried out aga...,cnn_dailymail,1,- Two recent attacks on Palestinians sparked I...
2,Former Kan. Junior College Coach Indicted (AP)...,ag_news,2,- Former Kansas junior college basketball coac...
3,The Ethiopian Airlines flight was travelling f...,xsum,3,- Ethiopian Airlines Flight ET500 was travelin...
4,"Minami Sanriku, Japan (CNN) -- A 60-year-old ...",cnn_dailymail,4,"- A 60‑year‑old man, Hiromitsu Shinkawa, was r..."


In [ ]:
MODEL_ID = "microsoft/bitnet-b1.58-2B-4T"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID
)

model = model.to("cuda")

model.eval()

print("BitNet loaded")
print("Device:", next(model.parameters()).device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

You have loaded a BitNet model on CPU and have a CUDA device available, make sure to set your model on a GPU device in order to run your model.


model.safetensors:   0%|          | 0.00/1.18G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/199 [00:00<?, ?B/s]

BitNet loaded
Device: cuda:0


In [ ]:
SYSTEM_PROMPT = """
Convert the provided English text into concise bullet points containing all materially important information.

Rules:

- Extract all important and independently useful points.
- The number of bullets must depend entirely on the information in the text.
- Do not use a fixed number of bullets.
- Use one bullet per distinct important point.
- Combine details that naturally belong together.
- Remove repetition, filler, and trivial information.
- Preserve important names, dates, numbers, quantities, comparisons, causes, conditions, decisions, and conclusions.
- Do not add information that is not supported by the input.
- Keep every bullet concise while preserving the original meaning.
- Return only bullet points.
- Start every bullet with "- ".
""".strip()

In [ ]:
def generate_bitnet(text, max_new_tokens=512):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": text
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        key: value.to("cuda")
        for key, value in inputs.items()
    }

    input_length = inputs["input_ids"].shape[-1]

    torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    generated_tokens = outputs[
        0,
        input_length:
    ]

    output_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    output_tokens = len(generated_tokens)

    tokens_per_second = (
        output_tokens / elapsed
        if elapsed > 0
        else 0
    )

    return {
        "output": output_text,
        "latency_seconds": elapsed,
        "output_tokens": output_tokens,
        "tokens_per_second": tokens_per_second
    }

In [ ]:
row = eval_df.iloc[0]

print("INPUT:\n")
print(row["text"])

print("\nREFERENCE:\n")
print(row["bullet_points"])

print("\nBITNET:\n")

result = generate_bitnet(
    row["text"]
)

print(result["output"])

print()
print(
    "Latency:",
    round(result["latency_seconds"], 2),
    "seconds"
)

print(
    "Tokens/sec:",
    round(result["tokens_per_second"], 2)
)

INPUT:

Powell: N. Korea Blast Not Nuclear Event The United States does not believe that a large explosion in North Korea was related to the communist country #39;s suspected nuclear weapons program, President Bush #39;s foreign policy advisers said Sunday.

REFERENCE:

- Powell says the large North Korean explosion was not a nuclear event.  
- U.S. officials do not believe the blast was linked to North Korea’s suspected nuclear weapons program.  
- The statement was made by President Bush’s foreign policy advisers on Sunday.

BITNET:



/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:641: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


- The United States does not believe a large explosion in North Korea was related to the country's suspected nuclear weapons program.
- President Bush's foreign policy advisers stated this on Sunday.

Latency: 10.33 seconds
Tokens/sec: 3.58


In [ ]:
RESULTS_FILE = "/content/bitnet_results.csv"

print(RESULTS_FILE)

/content/bitnet_results.csv


In [ ]:
if os.path.exists(RESULTS_FILE):

    existing_df = pd.read_csv(
        RESULTS_FILE
    )

    completed_ids = set(
        existing_df["example_id"]
        .astype(int)
        .tolist()
    )

else:

    completed_ids = set()


print(
    "Already completed:",
    len(completed_ids)
)

print(
    "Remaining:",
    len(eval_df) - len(completed_ids)
)

Already completed: 0
Remaining: 500


In [ ]:
for _, row in tqdm(
    eval_df.iterrows(),
    total=len(eval_df),
    desc="BitNet"
):

    example_id = int(
        row["example_id"]
    )

    if example_id in completed_ids:
        continue

    try:

        result = generate_bitnet(
            row["text"]
        )

        result_row = pd.DataFrame([
            {
                "example_id":
                    example_id,

                "source":
                    row["source"],

                "text":
                    row["text"],

                "reference":
                    row["bullet_points"],

                "output":
                    result["output"],

                "latency_seconds":
                    result["latency_seconds"],

                "output_tokens":
                    result["output_tokens"],

                "tokens_per_second":
                    result["tokens_per_second"]
            }
        ])

        result_row.to_csv(
            RESULTS_FILE,
            mode="a",
            header=not os.path.exists(
                RESULTS_FILE
            ),
            index=False
        )

        completed_ids.add(
            example_id
        )

    except Exception as e:

        print()
        print(
            f"Error on example {example_id}:"
        )
        print(e)

        # Clear temporary CUDA allocations
        gc.collect()
        torch.cuda.empty_cache()


print()
print(
    "Completed:",
    len(completed_ids)
)

print(
    "Saved:",
    RESULTS_FILE
)

BitNet:   0%|          | 0/500 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:641: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:6

In [ ]:
bitnet_results = pd.read_csv(
    RESULTS_FILE
)

print(
    "Results:",
    len(bitnet_results)
)

bitnet_results.head()